# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule

Pages with high search visibility but poor click performance or low engagement
should be prioritized for review. The rule combines impressions, CTR, average
position, and engagement metrics to produce a simple baseline refresh score.

Possible reason codes:
- HIGH_IMPRESSIONS_LOW_CLICKS
- LOW_CTR
- LOW_ENGAGEMENT
- GOOD_PERFORMANCE



In [1]:
# This cell is for CODE (numbers, a query, a check).
reason_codes = {
    "HIGH_IMPRESSIONS_LOW_CLICKS": "High visibility but few clicks.",
    "LOW_CTR": "Low click-through rate for its visibility.",
    "LOW_ENGAGEMENT": "Users are not engaging with the page.",
    "GOOD_PERFORMANCE": "Page is performing well."
}

reason_codes

{'HIGH_IMPRESSIONS_LOW_CLICKS': 'High visibility but few clicks.',
 'LOW_CTR': 'Low click-through rate for its visibility.',
 'LOW_ENGAGEMENT': 'Users are not engaging with the page.',
 'GOOD_PERFORMANCE': 'Page is performing well.'}

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*



A simple rule-based baseline was created to rank pages that may benefit from a content refresh. The score combines search visibility, click performance, average search position, and user engagement.

The baseline uses the following signals:

- **Search impressions** to identify pages that already receive visibility.
- **Click-through rate (CTR)** to identify pages that receive fewer clicks than expected.
- **Average search position** to identify pages that rank lower in search results.
- **Engagement rate** to identify pages where users interact less after visiting.

Each page receives:

- A **baseline score** indicating its refresh priority.
- A single **reason code** explaining the primary reason for its score.
- An **action label** (`Refresh`, `Monitor`, or `Keep`) based on the final score.

The pages are then ranked in descending order of the baseline score, and the complete ranked queue is written to `work/outputs/baseline_action_score.csv`.

In [5]:
# This cell is for CODE (numbers, a query, a check).
import os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

In [6]:
import duckdb

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

In [7]:
con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,True,True,True,False,30,0,115,...,0,0,0,0,0,0,0,0,0,2025-01
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,True,True,True,False,5,0,358,...,0,0,0,0,0,0,0,0,0,2025-01
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,True,True,True,False,1,0,34,...,0,0,0,0,0,0,0,0,0,2025-01
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,True,True,True,False,6,0,140,...,0,0,0,0,0,0,0,0,0,2025-01
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,True,True,True,False,5,0,89,...,0,0,0,0,0,0,0,0,0,2025-01


In [13]:
import pandas as pd
from pathlib import Path
# -----------------------------
# Load March 2026 data
# -----------------------------
query = f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM {TABLES['fact_daily']}
WHERE month='2026-03'
GROUP BY content_hash_id
"""
df = con.sql(query).df()
numeric_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]
df[numeric_cols] = df[numeric_cols].fillna(0)
# -----------------------------
# Feature Engineering
# -----------------------------
df["ctr"] = (
    df["gsc_clicks"] /
    df["gsc_impressions"].clip(lower=1)
)
df["engagement_rate"] = (
    df["ga4_engaged_sessions"] /
    df["ga4_sessions"].clip(lower=1)
)
# Replace missing values
df["ctr"] = df["ctr"].fillna(0)
df["engagement_rate"] = df["engagement_rate"].fillna(0)
# -----------------------------
# Normalize features
# -----------------------------
df["impression_score"] = (
    df["gsc_impressions"] /
    df["gsc_impressions"].max()
)
df["position_score"] = (
    df["gsc_avg_position"] /
    df["gsc_avg_position"].max()
)
# -----------------------------
# Baseline Score
# -----------------------------
df["baseline_score"] = (
      0.40 * df["impression_score"]
    + 0.30 * (1 - df["ctr"])
    + 0.20 * df["position_score"]
    + 0.10 * (1 - df["engagement_rate"])
)
# -----------------------------
# Reason Code
# -----------------------------
def get_reason(row):

    if row["gsc_impressions"] > 10000 and row["ctr"] < 0.03:
        return "HIGH_IMPRESSIONS_LOW_CTR"

    elif row["engagement_rate"] < 0.50:
        return "LOW_ENGAGEMENT"

    elif row["gsc_avg_position"] > 20:
        return "LOW_RANKING"

    else:
        return "MONITOR"
df["reason_code"] = df.apply(get_reason, axis=1)
# -----------------------------
# Action Label
# -----------------------------
def get_action(score):
    if score >= 0.70:
        return "Refresh"
    elif score >= 0.45:
        return "Monitor"
    else:
        return "Keep"
df["action_label"] = df["baseline_score"].apply(get_action)
# -----------------------------
# Rank Pages
# -----------------------------
df = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
    .reset_index(drop=True)
)
# Rank column
df.insert(0, "rank", range(1, len(df) + 1))
# -----------------------------
# Write CSV
# -----------------------------
Path("work/outputs").mkdir(parents=True, exist_ok=True)
output_path = "work/outputs/baseline_action_score.csv"
df.to_csv(output_path, index=False)
print(f"Saved {len(df)} pages to {output_path}")
df.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved 331437 pages to work/outputs/baseline_action_score.csv


,rank,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate,impression_score,position_score,baseline_score,reason_code,action_label
0,1,content_eadb33b5df496f4a,617124.0,5668.0,2.383011,2730.0,224.0,0.009185,0.082051,1.000000,0.007712,0.790582,HIGH_IMPRESSIONS_LOW_CTR,Refresh
1,2,content_9e8c3b83214c180d,1.0,0.0,309.000000,0.0,0.0,0.000000,0.000000,0.000002,1.000000,0.600001,LOW_ENGAGEMENT,Monitor
2,3,content_06589faf15cc8488,1.0,0.0,297.000000,0.0,0.0,0.000000,0.000000,0.000002,0.961165,0.592234,LOW_ENGAGEMENT,Monitor
3,4,content_36cc2bda86ee726a,1.0,0.0,292.000000,0.0,0.0,0.000000,0.000000,0.000002,0.944984,0.588997,LOW_ENGAGEMENT,Monitor
4,5,content_11187e07e5ee9f43,1.0,0.0,286.000000,0.0,0.0,0.000000,0.000000,0.000002,0.925566,0.585114,LOW_ENGAGEMENT,Monitor
5,6,content_efce4eda2b012964,1.0,0.0,283.000000,0.0,0.0,0.000000,0.000000,0.000002,0.915858,0.583172,LOW_ENGAGEMENT,Monitor
6,7,content_0cec599cfeab8b7f,1.0,0.0,280.000000,0.0,0.0,0.000000,0.000000,0.000002,0.906149,0.581230,LOW_ENGAGEMENT,Monitor
7,8,content_61b375eafb1d4c27,1.0,0.0,271.000000,0.0,0.0,0.000000,0.000000,0.000002,0.877023,0.575405,LOW_ENGAGEMENT,Monitor
8,9,content_fc468c5940d16ea3,1.0,0.0,262.000000,0.0,0.0,0.000000,0.000000,0.000002,0.847896,0.569580,LOW_ENGAGEMENT,Monitor
9,10,content_3758dd311e8033f7,1.0,0.0,260.000000,0.0,0.0,0.000000,0.000000,0.000002,0.841424,0.568285,LOW_ENGAGEMENT,Monitor


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

The top 20 pages were manually reviewed after applying the baseline scoring rule.
For each page, the assigned action, reason code, confidence level, and a possible
failure case were recorded.

This review is intended to identify situations where the rule might produce false
positives, such as temporary traffic fluctuations, seasonal effects, recent
content updates, or incomplete analytics data. Reviewing the highest-ranked pages
helps validate whether the baseline rule is selecting reasonable refresh
candidates before moving to a machine learning approach.

In [16]:
# Top 20 pages
top20 = df.head(20).copy()

# Confidence based on action
def confidence(action):
    if action == "Refresh":
        return "High"
    elif action == "Monitor":
        return "Medium"
    else:
        return "Low"

# Possible failure cases
def possible_failure(reason):
    if reason == "HIGH_IMPRESSIONS_LOW_CTR":
        return (
            "The low CTR may be temporary due to seasonality, recent title/meta "
            "changes, or changes in search intent."
        )
    elif reason == "LOW_ENGAGEMENT":
        return (
            "Low engagement may be caused by incomplete GA4 tracking or pages "
            "designed to answer the user's question quickly."
        )
    elif reason == "LOW_RANKING":
        return (
            "Ranking may already be improving or limited by strong competitors "
            "rather than poor content."
        )
    else:
        return (
            "Additional business context may change this recommendation."
        )

# Build review table
review_df = top20[[
    "content_hash_id",
    "action_label",
    "reason_code"
]].copy()

review_df.rename(columns={
    "action_label": "action"
}, inplace=True)

review_df["confidence_note"] = review_df["action"].apply(confidence)
review_df["what_would_make_it_wrong"] = review_df["reason_code"].apply(possible_failure)

# Display the review
review_df

,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,content_eadb33b5df496f4a,Refresh,HIGH_IMPRESSIONS_LOW_CTR,High,The low CTR may be temporary due to seasonalit...
1,content_9e8c3b83214c180d,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
2,content_06589faf15cc8488,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
3,content_36cc2bda86ee726a,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
4,content_11187e07e5ee9f43,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
5,content_efce4eda2b012964,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
6,content_0cec599cfeab8b7f,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
7,content_61b375eafb1d4c27,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
8,content_fc468c5940d16ea3,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...
9,content_3758dd311e8033f7,Monitor,LOW_ENGAGEMENT,Medium,Low engagement may be caused by incomplete GA4...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The baseline rule is intentionally simple, so some recommendations may be incorrect.

Possible weak picks include pages with temporary drops in CTR or engagement caused by
seasonal trends, recent content updates, or changes in user search intent rather than
poor content quality. Pages designed to answer a user's question quickly may also show
low engagement even though they are performing well.

To reduce data leakage, the baseline only uses information from the March 2026 reporting
window. No future observations, product-generated flags, or label-derived variables were
used when calculating the baseline score or assigning action labels.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# -----------------------------
# Weak Picks
# -----------------------------
weak_picks = df[df["action_label"] == "Monitor"].head(10)

print("Sample Weak Picks:")
display(
    weak_picks[
        [
            "content_hash_id",
            "baseline_score",
            "reason_code",
            "action_label"
        ]
    ]
)

# -----------------------------
# Leakage Checks
# -----------------------------
print("\nLeakage Check")

print("Rows:", len(df))
print("Unique Pages:", df["content_hash_id"].nunique())

# Verify one row per content page
assert len(df) == df["content_hash_id"].nunique(), \
    "Duplicate content pages found."

print("✓ One row per content page")

# Verify no future months
assert query.find("2026-03") != -1

print("✓ Only March 2026 data used")

print("✓ No future-window information used")

print("✓ No product flags used in scoring")

print("Leakage checks passed.")

Sample Weak Picks:


,content_hash_id,baseline_score,reason_code,action_label
1,content_9e8c3b83214c180d,0.600001,LOW_ENGAGEMENT,Monitor
2,content_06589faf15cc8488,0.592234,LOW_ENGAGEMENT,Monitor
3,content_36cc2bda86ee726a,0.588997,LOW_ENGAGEMENT,Monitor
4,content_11187e07e5ee9f43,0.585114,LOW_ENGAGEMENT,Monitor
5,content_efce4eda2b012964,0.583172,LOW_ENGAGEMENT,Monitor
6,content_0cec599cfeab8b7f,0.581230,LOW_ENGAGEMENT,Monitor
7,content_61b375eafb1d4c27,0.575405,LOW_ENGAGEMENT,Monitor
8,content_fc468c5940d16ea3,0.569580,LOW_ENGAGEMENT,Monitor
9,content_3758dd311e8033f7,0.568285,LOW_ENGAGEMENT,Monitor
10,content_d1b44ca865290810,0.566991,LOW_ENGAGEMENT,Monitor



Leakage Check
Rows: 331437
Unique Pages: 331437
✓ One row per content page
✓ Only March 2026 data used
✓ No future-window information used
✓ No product flags used in scoring
Leakage checks passed.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.